In [2]:
import os

folders = ['dags', 'scripts', 'logs', 'plugins', 'data']
base_dirs = 'airflow_buoi11'
for folder in folders:
    folder_path = os.path.join(base_dirs, folder)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Created folder: {folder_path}")
    else:
        print(f"Folder already exists: {folder_path}")

Created folder: airflow_buoi11\dags
Created folder: airflow_buoi11\scripts
Created folder: airflow_buoi11\logs
Created folder: airflow_buoi11\plugins
Created folder: airflow_buoi11\data


In [4]:
%%writefile airflow_buoi11/scripts/daily_revenue_etl.py
import os
import sys
import pandas as pd
from datetime import datetime, timedelta
from sqlalchemy import create_engine
from dotenv import load_dotenv
load_dotenv()

db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')
db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASS')

DB_CONNECTION = f'postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}' 

def get_daily_revenue(target_date):
    engine = create_engine(DB_CONNECTION)   
    try:
        query = f"""
            select 
                c.name as category_name,
                sum(p.amount) as total_revenue,
                '{target_date}'::date as revenue_date
            from payment p
            join rental r on p.rental_id = r.rental_id
            join inventory i on r.inventory_id = i.inventory_id
            join film_category fc on fc.film_id = i.film_id
            join category c on fc.category_id = c.category_id
            where DATE(p.payment_date) = '{target_date}'
            group by c.name
            order by total_revenue desc;
        """

        df = pd.read_sql_query(query, con=engine)
        if not df.empty:
            print(f"Dữ liệu ngày {target_date} đã được tải thành công.")
            df.to_sql('daily_revenue_report', con=engine, if_exists='append', index=False)
            print("Dữ liệu đã được chèn vào bảng 'daily_revenue_report'.")
        else:
            print(f"Không có dữ liệu cho ngày {target_date}.")
            
    except Exception as e:
        print(f"Lỗi khi lấy dữ liệu hoặc chèn vào bảng: {e}")
        sys.exit(1)
        
if __name__ == "__main__":
    if len(sys.argv) > 1:
        print("Usage: python script.py <target_date>")
        sys.exit(1)
        target_date = sys.argv[1]

    else:
        target_date = (datetime.now() - timedelta(1)).strftime('%Y-%m-%d')
    get_daily_revenue(target_date)

Writing airflow_buoi11/scripts/daily_revenue_etl.py


In [ ]:
%%writefile airflow_buoi11/dags/daily_revenue_etl_dag.py
from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.empty import EmptyOperator
from datetime import datetime, timedelta

# Define default arguments for the DAG
default_args = {    
    'owner': 'data_engineer',
    'depends_on_past': False,
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

# Define the DAG
with DAG(
    dag_id='daily_revenue_etl_dag',
    default_args=default_args,
    description='A DAG to run daily revenue ETL script',
    schedule_interval='@daily', 
    start_date=datetime(2024, 1, 1),
    catchup=False,
    tags=['etl', 'daily_revenue'],
) as dag:

    task_start = EmptyOperator(task_id='start_pipeline')

    task_run_etl = BashOperator(
        task_id='run_daily_revenue_etl',
        bash_command='python /opt/airflow/scripts/daily_revenue_etl.py {{ ds }}',
    )

    task_end = EmptyOperator(task_id='end_task')

    # Define task dependencies
    task_start >> task_run_etl >> task_end

Writing airflow_buoi11/dags/daily_revenue_etl_dag.py
